# 曙光 v3 說話參考 → Qwen3-TTS 克隆（Colab 免費 GPU）

**用途：** 用 `dawn-natural-female-v3-speak.wav` 在 **Google Colab 免費 T4** 上跑開源神經克隆，產出 2–3 句國語試聽 wav。  
**產品狀態：** 本 notebook **不會**也不應標 `AVAILABLE`。聽感通過前產品仍是 `BLOCKED_NEED_GPU` / MUTED。  
**禁止：** Gemini Live 當產品聲、瀏覽器 `speechSynthesis`、付費 TTS API。

| 項目 | 值 |
|------|-----|
| 引擎 | **Qwen3-TTS-12Hz-0.6B-Base**（預設；可改 1.7B） |
| 授權 | Apache-2.0（上游 QwenLM/Qwen3-TTS） |
| 硬體 | Runtime → **GPU**（T4 足夠；勿用純 CPU） |
| 參考 | **自動下載**倉庫 `dawn-natural-female-v3-speak.wav`（手機不必上傳） |

> Colab 會要求 **Google 登入**；免費額度有限、可能排隊或斷線。這一步只為「解鎖克隆樣品」；日常產品推論以後仍需自架／本機 GPU 節點。


## 0. 誠實聲明（必讀）

1. 產出 wav = **候選試聽**，不是產品上線。  
2. v3 來源是廣告配音 + Demucs；可能殘 BGM／廣告腔 — 克隆可能繼承。  
3. 通過抑揚頓挫聽感後，才可由維護者把 `DawnNeuralTts` 標 `READY`／文件寫 `AVAILABLE`。  
4. **不要**在本 notebook 接 Gemini Live。


## 1. 檢查 GPU（必須有 CUDA）


In [ ]:
# Runtime → Change runtime type → Hardware accelerator: GPU (T4)
import torch, sys
print("python", sys.version)
print("torch", torch.__version__)
print("cuda_available", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("STOP: 沒有 GPU。請改 Runtime → GPU 後重跑。不要用 CPU 硬撐產品路徑。")
print("gpu", torch.cuda.get_device_name(0))
print("vram_gb", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## 2. 安裝（免費開源；約數分鐘）


In [ ]:
# Apache-2.0 Qwen3-TTS. Pin lightly; Colab images drift.
%pip -q install -U qwen-tts soundfile librosa accelerate
# flash-attn optional; sdpa fallback is fine on T4
import qwen_tts, soundfile, torch
print("qwen_tts ok", getattr(qwen_tts, "__file__", qwen_tts))


## 3. 取得 v3 參考 wav（**自動下載 · 手機免上傳**）

從 GitHub `main` 拉倉庫資產（已合併 PR #35）。  
預期 SHA-256：`c42a877c6af9cb48384e4b9a8bf3d24aac137107faf49d5201955df69398cf1d`

若自動下載失敗，再用左側資料夾手動上傳同名檔。


In [ ]:
from pathlib import Path
import hashlib, urllib.request

REF_URL = (
    "https://raw.githubusercontent.com/NL13741619-commits/homework-helper/"
    "main/app/assets/dawn-voice/dawn-natural-female-v3-speak.wav"
)
EXPECTED_SHA256 = "c42a877c6af9cb48384e4b9a8bf3d24aac137107faf49d5201955df69398cf1d"
ref_path = Path("/content/dawn-natural-female-v3-speak.wav")

print("downloading", REF_URL)
urllib.request.urlretrieve(REF_URL, ref_path)
sha = hashlib.sha256(ref_path.read_bytes()).hexdigest()
print("saved", ref_path, "bytes", ref_path.stat().st_size)
print("sha256", sha)
if sha != EXPECTED_SHA256:
    raise SystemExit(f"STOP: sha256 mismatch\n got {sha}\n want {EXPECTED_SHA256}")
print("OK: auto-download v3 speak reference ready (no manual upload)")


## 4. 載入模型（預設 0.6B-Base · T4 友善）

可改成 `Qwen/Qwen3-TTS-12Hz-1.7B-Base`（較吃 VRAM）。  
**替代引擎：** CosyVoice2（Apache-2.0）也可；本 notebook 以 Qwen3-TTS 為準（與 `app/services/aivault-voice` 同系）。


In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"  # or 1.7B-Base
model = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
)
print("loaded", MODEL_ID)


## 5. 克隆 2–3 句國語試聽

`ref_text`：若你知道原廣告逐字稿可填；未知則用 `x_vector_only_mode=True`（品質可能較差，誠實降級）。


In [ ]:
import soundfile as sf
from pathlib import Path

OUT = Path("/content/dawn-clone-out")
OUT.mkdir(exist_ok=True)

# Optional: paste known transcript of the ~12s ad VO; else leave None
REF_TEXT = None  # e.g. "……"

texts = [
    ("01-greeting", "你好，我是曙光。很高興認識你。"),
    ("02-question", "今天想先聽哪一段？可以慢一點再說一次嗎？"),
    ("03-calm", "沒關係，我們一步一步來。你先深呼吸，再告訴我卡在哪裡。"),
]

# If no transcript, x_vector_only_mode=True
use_xvec = REF_TEXT is None
paths = []
for key, text in texts:
    kwargs = dict(
        text=text,
        language="Chinese",
        ref_audio=str(ref_path),
    )
    if use_xvec:
        kwargs["x_vector_only_mode"] = True
    else:
        kwargs["ref_text"] = REF_TEXT
    wavs, sr = model.generate_voice_clone(**kwargs)
    out = OUT / f"dawn-v3-qwen3-{key}.wav"
    sf.write(out, wavs[0], sr)
    paths.append(out)
    print("wrote", out, "sr", sr)

print("DONE — download these wavs; ear-check 抑揚頓挫 before any AVAILABLE claim")
for p in paths:
    print(" -", p)


## 6. 試聽 + 下載


In [ ]:
from IPython.display import Audio, display
from google.colab import files
for p in sorted(OUT.glob("*.wav")):
    print(p.name)
    display(Audio(str(p)))
# Zip for one download
import shutil
shutil.make_archive("/content/dawn-v3-qwen3-clones", "zip", OUT)
files.download("/content/dawn-v3-qwen3-clones.zip")
print("請把 zip 交回維護者放入 /workspace/reports/dawn-voice/neural-poc/colab-out/")


## 7. 聽感 checklist（人工）

- [ ] 音色像 v3 女聲（允許廣告腔）
- [ ] 不是電音／機器人腔
- [ ] 疑問句有上揚、句子有停頓（抑揚頓挫）
- [ ] 沒有把殘 BGM 放大成伴奏床
- [ ] **全部通過前：產品仍 NOT AVAILABLE**

下一步（通過後）：自架／本機 GPU 跑 `app/services/aivault-voice`，接 `dawn-neural-tts.js`。Colab 只負責解鎖樣品，不適合當 7×24 產品推論。
